In [1]:

!pip install langchain qdrant-client -q
!pip install -U langchain-community


[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\nguye\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\nguye\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install openpyxl
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\nguye\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\nguye\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
!pip install sentence_transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: C:\Users\nguye\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
import pandas as pd
from langchain.document_loaders import TextLoader
from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain.vectorstores import Qdrant
from qdrant_client import QdrantClient

In [ ]:
# Cài đặt thông tin truy cập vector database
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
QDRANT_URL = userdata.get("QDRANT_URL")
# EMBEDDINGS_MODEL_NAME="sentence-transformers/LaBSE"
EMBEDDINGS_MODEL_NAME="intfloat/multilingual-e5-base"
HUGGINGFACE_API_KEY= userdata.get("HUGGINGFACE_API_KEY")
COLLECTION_NAME = "ITUS_e5_R_800"

In [105]:
from langchain.embeddings.base import Embeddings
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import torch


class CustomHuggingFaceEmbeddings(Embeddings):
    def __init__(self, model_name: str):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def embed_documents(self, texts):
        """
        Embed a list of documents (or queries).
        """
        return [self._embed_text(text) for text in texts]

    def embed_query(self, text):
        """
        Embed a single query.
        """
        return self._embed_text(text)

    def _embed_text(self, text):
        """
        Tokenize, encode, and pool the text into an embedding.
        """
        inputs = self.tokenizer(
            text,
            max_length=512,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        outputs = self.model(**inputs)
        # Apply mean pooling
        embeddings = self._mean_pool(outputs.last_hidden_state, inputs['attention_mask'])
        # Normalize embeddings
        return F.normalize(embeddings, p=2, dim=1).squeeze(0).tolist()

    @staticmethod
    def _mean_pool(last_hidden_state, attention_mask):
        """
        Mean pooling: mask out padding tokens and average over remaining tokens.
        """
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1)


# Khởi tạo embedding model
LOCAL_MODEL_PATH = "models/multilingual-e5-base"

embeddings = CustomHuggingFaceEmbeddings(model_name=LOCAL_MODEL_PATH)

# Test với văn bản
texts = [
    "query: how much protein should a female eat",
    "passage: As a general guideline, the CDC's average requirement of protein for women ages 19 to 70 is 46 grams per day."
]

embedded_texts = embeddings.embed_documents(texts)
print("Embedding vector:", embedded_texts)


Embedding vector: [[-0.004460367374122143, 0.054887931793928146, 0.0004126062267459929, 0.031007615849375725, 0.028034169226884842, -0.06244392693042755, -0.01671105995774269, -0.02610591985285282, 0.01724051497876644, 0.026319123804569244, 0.012052218429744244, 0.008626500144600868, 0.1347959339618683, 0.020417511463165283, -0.03065212443470955, -0.0037135849706828594, 0.02494938112795353, 0.00643831817433238, 0.06547459959983826, -0.01139923371374607, 0.05026664957404137, -0.023976678028702736, 0.021874677389860153, -0.03405801206827164, 0.02256707474589348, -0.02258891984820366, -0.010900823399424553, 0.03283685818314552, -0.01742355339229107, 0.06272925436496735, 0.028554826974868774, -0.011197905987501144, -0.0028248229064047337, 0.02622869424521923, 0.03997163474559784, -0.004212149418890476, -0.012988160364329815, -0.03043217957019806, 0.022728553041815758, 0.009146396070718765, -0.004415902309119701, 0.012234729714691639, 0.018140314146876335, -0.06056125462055206, 0.0444186069

In [106]:
from qdrant_client import QdrantClient

# Khởi tạo kết nối Qdrant API
qdrant_client = QdrantClient(
    url=QDRANT_URL,  # Đường dẫn URL của Qdrant
    api_key=QDRANT_API_KEY  # API key của Qdrant
)

In [107]:
# from langchain.embeddings import HuggingFaceEmbeddings
# from sentence_transformers import SentenceTransformer
# import os

# def load_or_download_model(model_name, local_path=None):
#     """
#     Tải hoặc sử dụng model SentenceTransformer từ cục bộ nếu có.

#     Args:
#     - model_name (str): Tên model trên Hugging Face.
#     - local_path (str, optional): Đường dẫn cục bộ để kiểm tra model.

#     Returns:
#     - model (HuggingFaceEmbeddings): Embeddings được khởi tạo.
#     """
#     if local_path and os.path.exists(local_path):
#         print(f"Loading model from local path: {local_path}")
#         model = HuggingFaceEmbeddings(model_name=local_path)
#     else:
#         print(f"Downloading model from Hugging Face: {model_name}")
#         model = HuggingFaceEmbeddings(model_name=model_name)
#     return model

# # Tên model trên Hugging Face
# MODEL_NAME = 'sentence-transformers/LaBSE'
# # Đường dẫn cục bộ để lưu hoặc tải model
# LOCAL_MODEL_PATH = './models/LaBSE'

# # Tải hoặc sử dụng model từ cục bộ
# embeddings = load_or_download_model(MODEL_NAME, local_path=LOCAL_MODEL_PATH)

# # Sử dụng embeddings để tạo vector cho danh sách câu
# sentences = ["This is an example sentence", "Each sentence is converted"]
# vectors = [embeddings.embed_query(sentence) for sentence in sentences]

# # In kết quả vector
# print(vectors)


In [108]:
# COLLECTION_NAME = "ITUS_LaBSE_600" 

In [109]:
# # Kiểm tra nếu collection đã tồn tại
# if COLLECTION_NAME in [collection.name for collection in qdrant_client.get_collections().collections]:
#     print(f"Collection '{COLLECTION_NAME}' đã tồn tại. Sử dụng collection này để đánh giá.")
#     vector_store = Qdrant(client=qdrant_client, collection_name=COLLECTION_NAME, embeddings=embeddings)

In [110]:
import pandas as pd
import os
import numpy as np


# Tính DCG@K
def dcg_at_k(predicted_docs, true_docs, k):
    dcg = 0
    for rank, doc in enumerate(predicted_docs[:k], start=1):
        # Gán độ liên quan là 1 nếu tài liệu đúng, 0 nếu không đúng
        relevance = 1 if doc in true_docs else 0
        dcg += relevance / np.log2(rank + 1)
    return dcg

# Tính IDCG@K (Ideal DCG)
def idcg_at_k(true_docs, k):
    # Sắp xếp danh sách đúng theo thứ tự giảm dần độ liên quan (tất cả đều có độ liên quan là 1)
    ideal_docs = true_docs[:k]  # Sắp xếp các tài liệu đúng trong ground truth theo độ liên quan
    idcg = 0
    for rank, _ in enumerate(ideal_docs, start=1):
        idcg += 1 / np.log2(rank + 1)  # Vì độ liên quan của tất cả các mục đúng đều là 1
    return idcg



def read_and_evaluate_retrieval(excel_path, vector_store, k=5, output_path="Evaluation_Results.xlsx"):
    if os.path.exists(excel_path):
        try:
            # Đọc dữ liệu từ Excel
            evaluation_data = pd.read_excel(excel_path, sheet_name=0)

            # Kiểm tra sự tồn tại của các cột cần thiết
            required_columns = ['question', 'expected_output', 'ground_truth']
            for col in required_columns:
                if col not in evaluation_data.columns:
                    raise ValueError(f"Cột '{col}' không tồn tại trong tập dữ liệu.")

            # Trích xuất dữ liệu từ các cột
            example_queries = evaluation_data['question'].fillna("").tolist()
            ground_truth_docs = (
                evaluation_data['ground_truth']
                .fillna("")
                .apply(lambda x: str(x).split('|'))
                .tolist()
            )

            # Khởi tạo danh sách để lưu kết quả chi tiết
            detailed_results = []

            # Đánh giá hệ thống retrieval
            all_precision = []
            all_recall = []
            all_rr = []  # Reciprocal Rank
            all_ap = [] 
            all_ndcg = [] 

            for idx, (query, true_docs) in enumerate(zip(example_queries, ground_truth_docs)):
                # Tìm kiếm tương tự
                results = vector_store.similarity_search(query, k=k)

                # Lấy giá trị 'source' từ metadata
                predicted_docs = [res.metadata['source'] for res in results]
                predicted_docs_unique = list(set(predicted_docs))
                # Tính Precision@K
                hits = sum([predicted_docs.count(doc) for doc in true_docs])
                precision = hits / k
                all_precision.append(precision)

                # Tính Recall@K
                hits_unique = len(set(predicted_docs) & set(true_docs))
                recall = hits_unique / len(true_docs) if true_docs else 0
                all_recall.append(recall)

                # Tính Reciprocal Rank (RR)
                rr = 0
                for rank, doc in enumerate(predicted_docs, start=1):
                    if doc in true_docs:
                        rr = 1 / rank
                        break
                all_rr.append(rr)

                # Tính AP@K cho truy vấn hiện tại
                relevant_docs = [doc for doc in predicted_docs_unique if doc in true_docs]
                ap_k = 0
                num_relevant = len(relevant_docs)
                if num_relevant > 0:
                    for rank, doc in enumerate(predicted_docs_unique[:k], start=1):
                        if doc in true_docs:
                            precision_at_k = sum([doc in predicted_docs_unique[:rank] for doc in true_docs]) / rank
                            ap_k += precision_at_k
                    ap_k /= num_relevant
                all_ap.append(ap_k)
                

                # Tính DCG@K
                dcg_k = dcg_at_k(predicted_docs_unique, true_docs, k)

                # Tính IDCG@K
                idcg_k = idcg_at_k(true_docs, k)

                # Tính NDCG@K
                ndcg_k = dcg_k / idcg_k if idcg_k > 0 else 0  # Tránh chia cho 0 nếu IDCG@K = 0
                all_ndcg.append(ndcg_k)


                # Lưu kết quả chi tiết
                detailed_results.append({
                    'Query': query,
                    'Ground Truth': true_docs,
                    'Predicted': predicted_docs,
                    'Precision@K': precision,
                    'Recall@K': recall,
                    'Reciprocal Rank': rr,
                    'AP@K': ap_k,
                    'NDCG@K': ndcg_k
                })


            # Tính trung bình Precision@K, Recall@K, MRR, MAP@K
            mean_precision = np.mean(all_precision)
            mean_recall = np.mean(all_recall)
            mean_rr = np.mean(all_rr)
            mean_ap = np.mean(all_ap)
            mean_ndcg = np.mean(all_ndcg)

            # Lưu kết quả vào DataFrame và xuất Excel
            results_df = pd.DataFrame(detailed_results)
            results_df.to_excel(output_path, index=False)

            # In kết quả ra màn hình
            print(f"\nĐánh giá hoàn tất! Kết quả đã được lưu tại: {output_path}\n")
            print(f"Mean Precision@{k}: {mean_precision:.4f}")
            print(f"Mean Recall@{k}: {mean_recall:.4f}")
            print(f"Mean Reciprocal Rank (MRR): {mean_rr:.4f}")
            print(f"Mean Average Precision (MAP@{k}): {mean_ap:.4f}")
            print(f"Mean NDCG@{k}: {mean_ndcg:.4f}\n")
            print("Chi tiết kết quả:\n")
            print(results_df[['Query', 'Ground Truth', 'Predicted', 'Precision@K', 'Recall@K', 'Reciprocal Rank', 'AP@K', 'NDCG@K']].to_string(index=False))

        except Exception as e:
            print(f"Đã xảy ra lỗi: {e}")
    else:
        print(f"Tập tin '{excel_path}' không tồn tại.")


In [111]:
COLLECTION_NAME = "ITUS_e5_1000"
# Kiểm tra nếu collection đã tồn tại
if COLLECTION_NAME in [collection.name for collection in qdrant_client.get_collections().collections]:
    print(f"Collection '{COLLECTION_NAME}' đã tồn tại. Sử dụng collection này để đánh giá.")
    vector_store = Qdrant(client=qdrant_client, collection_name=COLLECTION_NAME, embeddings=embeddings)

Collection 'ITUS_e5_1000' đã tồn tại. Sử dụng collection này để đánh giá.


In [112]:
read_and_evaluate_retrieval(
    excel_path="QA_data (1).xlsx",  # Đường dẫn tập tin dữ liệu
    vector_store=vector_store,     # Hệ thống vector_store
    k=5,                           # Top K
    output_path="Evaluation_Results.xlsx"  # Tập tin kết quả
)

print(COLLECTION_NAME)


Đánh giá hoàn tất! Kết quả đã được lưu tại: Evaluation_Results.xlsx

Mean Precision@5: 0.7806
Mean Recall@5: 0.9676
Mean Reciprocal Rank (MRR): 0.8980
Mean Average Precision (MAP@5): 0.7691
Mean NDCG@5: 0.8137

Chi tiết kết quả:

                                                                                                                                                Query                                                   Ground Truth                                                    Predicted  Precision@K  Recall@K  Reciprocal Rank     AP@K   NDCG@K
                                                                                  Tổng số tín chỉ bắt buộc của ngành Hệ thống thông tin là bao nhiêu?                                                     [HTTT.txt]           [HTTT.txt, HTTT.txt, HTTT.txt, TTNT.txt, KTPM.txt]          0.6  1.000000         1.000000 0.333333 0.500000
                                                                                       Tên văn bằng ngành